# Radar Cidadão — completar 95 para ≥100 QA válidas

**Correção incremental sem repetir as 100 perguntas já produzidas.** Usa o `corpus_qa.jsonl` exportado no Google Drive, seleciona apenas novos segmentos oficiais e gera QAs adicionais com o Qwen2.5-14B-Instruct e os filtros v8.3.

- Não altera nem apaga a pasta anterior `entrega_GS_19nov`.
- Cria `entrega_GS_19nov_corrigida` somente quando alcançar 100 QAs válidas.
- Reutiliza `corpus_texto.jsonl` (306) e `corpus_chat.jsonl` (31, incluindo 11 recusas).
- Gera planilha nova para revisão humana, relatório PDF **rascunho** e ZIP **rascunho**.
- **Não marca revisão humana nem fidelidade como concluída.**

Ative GPU no Colab. Execute as células em ordem; não execute a geração original das 100 QAs.

In [ ]:
# 1. Montar Drive, ler a entrega anterior e preparar verificadores locais
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, sys, shutil, subprocess, importlib, math, random
BASE=Path('/content/drive/MyDrive/radar-cidadao')
ANTERIOR=BASE/'entrega_GS_19nov'
NOVO=BASE/'entrega_GS_19nov_corrigida'
REPO=Path('/content/radar-cidadao')
PROJETO=REPO/'nlp-fabrica-corpus'
for nome in ('corpus_texto.jsonl','corpus_qa.jsonl','corpus_chat.jsonl'):
    assert (ANTERIOR/nome).is_file(), f'Falta arquivo anterior no Drive: {ANTERIOR/nome}'
assert (PROJETO/'src'/'fabrica_corpus'/'geracao.py').exists(), ('Repositório não encontrado no Colab. Execute a célula de configuração do notebook original (sem executar a geração), ou clone o repositório.')
src=str(PROJETO/'src')
if src not in sys.path: sys.path.insert(0,src)
sys.path.insert(0,'/content')
# Helpers de exportação correspondentes à última entrega. Não modificam o GitHub.
Path('/content/fechar_entrega.py').write_text('"""Radar Cidadao: exportacao conservadora de dados existentes, verificacao e relatorio.\nNao aprova automaticamente fidelidade; nao cria exemplos inexistentes.\n"""\nfrom __future__ import annotations\nimport csv, hashlib, json, math, random, re, statistics, unicodedata\nfrom collections import Counter\nfrom datetime import datetime\nfrom pathlib import Path\n\nMINIMOS = {\'texto\':50,\'qa\':100,\'chat\':30}\n\ndef _campo(o, nome, default=None): return o.get(nome, default) if isinstance(o,dict) else getattr(o,nome,default)\ndef _norm(s):\n    t=unicodedata.normalize(\'NFKD\',str(s)).casefold()\n    return re.sub(r\'\\s+\',\' \',re.sub(r\'[^a-z0-9 ]\',\' \', \'\'.join(c for c in t if not unicodedata.combining(c)))).strip()\ndef _chave(e, tipo):\n    if tipo==\'texto\': s=e[\'text\']\n    elif tipo==\'qa\': s=e[\'instruction\']\n    else: s=\'\\n\'.join(m[\'role\']+\':\'+m[\'content\'] for m in e[\'messages\'] if m[\'role\']!=\'system\')\n    return hashlib.sha256(_norm(s).encode()).hexdigest()\ndef _conteudo(seg):\n    for nome in (\'texto\',\'text\',\'conteudo\'):\n        v=_campo(seg,nome)\n        if isinstance(v,str) and v.strip(): return v.strip()\n    return \'\'\ndef normalizar_texto(segs):\n    saida=[]\n    for seg in segs:\n        fonte=_campo(seg,\'fonte\'); txt=_conteudo(seg)\n        if fonte and txt:\n            saida.append({\'text\':txt,\'fonte\':str(fonte),\'segmento_id\':str(_campo(seg,\'id\',\'\'))})\n    return saida\n\ndef normalizar_qa(itens):\n    resultado=[]\n    for e in itens:\n        pergunta=_campo(e,\'pergunta\',\'\'); resposta=_campo(e,\'resposta\',\'\'); fonte=_campo(e,\'fonte\',\'\')\n        if pergunta and resposta and fonte:\n            resultado.append({\'instruction\':pergunta.strip(),\'input\':\'\',\'output\':resposta.strip(), \'fonte\':str(fonte), \'segmento_id\':str(_campo(e,\'segmento_id\',\'\'))})\n    return resultado\n\ndef normalizar_chat(itens):\n    resultado=[]\n    for e in itens:\n        ms=_campo(e,\'mensagens\',[]); fonte=_campo(e,\'fonte\')\n        if not fonte or not isinstance(ms,list): continue\n        mensagens=[{\'role\':_campo(m,\'role\',\'\'), \'content\':_campo(m,\'content\',\'\')} for m in ms]\n        resultado.append({\'messages\':mensagens,\'fonte\':str(fonte),\'tipo\':str(_campo(e,\'tipo\',\'chat\')), \'segmento_id\':str(_campo(e,\'segmento_id\',\'\') or \'\')})\n    return resultado\n\ndef conferir_formato(e,tipo):\n    if not isinstance(e,dict) or not isinstance(e.get(\'fonte\'),str) or not e[\'fonte\'].strip(): return \'fonte ausente\'\n    if tipo==\'texto\':\n        if not isinstance(e.get(\'text\'),str) or len(e[\'text\'].strip())<30:return \'texto curto/ausente\'\n    if tipo==\'qa\':\n        if not all(isinstance(e.get(k),str) for k in [\'instruction\',\'input\',\'output\']):return \'campos QA ausentes\'\n        if len(e[\'instruction\'].strip())<12 or len(e[\'output\'].strip())<12:return \'pergunta/resposta curta\'\n    if tipo==\'chat\':\n        ms=e.get(\'messages\')\n        if not isinstance(ms,list):return \'mensagens ausentes\'\n        papeis=[m.get(\'role\') for m in ms if isinstance(m,dict) and m.get(\'role\')!=\'system\']\n        if len(papeis)<4 or len(papeis)%2 or papeis != [\'user\',\'assistant\']*(len(papeis)//2): return \'menos de 2 trocas ou papeis invalidos\'\n        if any(not isinstance(m,dict) or not isinstance(m.get(\'content\'),str) or not m[\'content\'].strip() or m.get(\'role\') not in (\'system\',\'user\',\'assistant\') for m in ms):return \'mensagem vazia/papel invalido\'\n    return None\n\ndef filtrar(itens,tipo):\n    validos=[]; duplicatas=0; invalidos=[]; vistos=set()\n    for n,e in enumerate(itens,1):\n        problema=conferir_formato(e,tipo)\n        if problema:\n            invalidos.append({\'linha\':n,\'problema\':problema});continue\n        k=_chave(e,tipo)\n        if k in vistos: duplicatas+=1;continue\n        vistos.add(k);validos.append(e)\n    return validos,duplicatas,invalidos\n\ndef salvar_jsonl(p,itens):\n    with Path(p).open(\'w\',encoding=\'utf-8\') as f:\n        for e in itens:f.write(json.dumps(e,ensure_ascii=False)+\'\\n\')\n\ndef ler_jsonl(p):\n    itens=[]\n    with Path(p).open(encoding=\'utf-8\') as f:\n        for n,l in enumerate(f,1):\n            if l.strip():\n                try:itens.append(json.loads(l))\n                except json.JSONDecodeError as ex:raise ValueError(f\'{p}: linha {n}: {ex}\') from ex\n    return itens\n\ndef estimar_tokens(e):return len(re.findall(r\'\\w+|[^\\w\\s]\',json.dumps(e,ensure_ascii=False)))\n\ndef exportar(destino,segs,qa,chat,recusas=(),fonte_documentos=None):\n    destino=Path(destino);destino.mkdir(parents=True,exist_ok=True)\n    brutos={\'texto\':normalizar_texto(segs),\'qa\':normalizar_qa(qa),\'chat\':normalizar_chat(list(chat)+list(recusas))}\n    metr={};amostras={}\n    for tipo,original in brutos.items():\n        validos,dup,invalidos=filtrar(original,tipo)\n        salvar_jsonl(destino/f\'corpus_{tipo}.jsonl\',validos)\n        metr[tipo]={\'quantidade\':len(validos),\'minimo\':MINIMOS[tipo],\'quantidade_atingida\':len(validos)>=MINIMOS[tipo], \'media_tokens_aprox\':round(statistics.mean(map(estimar_tokens,validos)),1) if validos else 0,\'duplicatas_removidas\':dup,\'invalidos_removidos\':len(invalidos),\'detalhes_invalidos\':invalidos[:40]}\n        amostras[tipo]=random.Random(2026).sample(validos,min(5,len(validos)))\n    metr[\'recusas_validas\']=sum(1 for e in ler_jsonl(destino/\'corpus_chat.jsonl\') if e.get(\'tipo\')==\'recusa\')\n    metr[\'recusas_atingidas\']=metr[\'recusas_validas\']>=10\n    metr[\'revisao_humana_confirmada\']=False\n    metr[\'fontes_documentais_conferidas\']=False\n    metr[\'observacao\']=\'Contagem e estrutura verificadas por codigo; fidelidade a fontes e revisao de >=20% exigem verificacao humana.\'\n    metr[\'pronto_para_entrega\']=all(metr[k][\'quantidade_atingida\'] for k in MINIMOS) and metr[\'recusas_atingidas\'] and metr[\'revisao_humana_confirmada\'] and metr[\'fontes_documentais_conferidas\']\n    (destino/\'estatisticas.json\').write_text(json.dumps(metr,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    (destino/\'amostras_5_por_corpus.json\').write_text(json.dumps(amostras,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    criar_revisao(destino)\n    return metr\n\ndef criar_revisao(destino):\n    destino=Path(destino); p=destino/\'revisao_humana_20_porcento.csv\'\n    # Criar sem sobrescrever uma revisao ja preenchida\n    if p.exists():return p\n    qa=ler_jsonl(destino/\'corpus_qa.jsonl\'); chats=ler_jsonl(destino/\'corpus_chat.jsonl\')\n    rng=random.Random(2026); selecionados=[]\n    for tipo,lista in ((\'qa\',qa),(\'chat\',chats)):\n        n=math.ceil(.2*len(lista));selecionados.extend((tipo,i,e) for i,e in enumerate(rng.sample(lista,n),1))\n    with p.open(\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n        w=csv.writer(f,delimiter=\';\');w.writerow([\'tipo\',\'numero_amostra\',\'fonte\',\'pergunta_ou_dialogo\',\'resposta\',\'apoio_na_fonte_S_N\',\'responde_S_N\',\'seguranca_S_N\',\'revisor\',\'observacoes\'])\n        for tipo,i,e in selecionados:\n            texto=e[\'instruction\'] if tipo==\'qa\' else \'\\n\'.join(m[\'role\']+\': \'+m[\'content\'] for m in e[\'messages\'] if m[\'role\']!=\'system\')\n            resposta=e[\'output\'] if tipo==\'qa\' else \'\'\n            w.writerow([tipo,i,e[\'fonte\'],texto,resposta,\'\',\'\',\'\',\'\',\'\'])\n    return p\n\ndef auditar_pasta(pasta):\n    pasta=Path(pasta); ret={}\n    for tipo in MINIMOS:\n        p=pasta/f\'corpus_{tipo}.jsonl\'\n        if not p.exists(): ret[tipo]={\'erro\':\'arquivo nao existe\'};continue\n        data=ler_jsonl(p);validos,dup,invalidos=filtrar(data,tipo)\n        ret[tipo]={\'quantidade\':len(data),\'quantidade_valida\':len(validos),\'minimo\':MINIMOS[tipo],\'duplicatas\':dup,\'invalidos\':invalidos,\'ok\':len(data)==len(validos) and len(validos)>=MINIMOS[tipo]}\n    if (pasta/\'corpus_chat.jsonl\').exists():\n        ret[\'recusas\']=sum(1 for e in ler_jsonl(pasta/\'corpus_chat.jsonl\') if e.get(\'tipo\')==\'recusa\')\n    return ret\n\nif __name__==\'__main__\':\n    import argparse\n    a=argparse.ArgumentParser();a.add_argument(\'pasta\');args=a.parse_args()\n    print(json.dumps(auditar_pasta(args.pasta),ensure_ascii=False,indent=2))\n',encoding='utf-8')
Path('/content/relatorio_pdf.py').write_text('"""Gera relatorio tecnico de no maximo 6 paginas, sem inventar aprovacao humana."""\nimport json\nfrom pathlib import Path\nfrom reportlab.lib import colors\nfrom reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle\nfrom reportlab.lib.enums import TA_CENTER\nfrom reportlab.lib.pagesizes import A4\nfrom reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, KeepTogether\n\n\ndef construir(pasta):\n    pasta=Path(pasta);met=json.loads((pasta/\'estatisticas.json\').read_text(encoding=\'utf-8\'))\n    p=pasta/\'relatorio_GS_Radar_Cidadao_RASCUNHO.pdf\'\n    st=getSampleStyleSheet();st.add(ParagraphStyle(name=\'CorpoGS\',parent=st[\'BodyText\'],fontSize=10,leading=15,spaceAfter=10))\n    st.add(ParagraphStyle(name=\'TitGS\',parent=st[\'Title\'],textColor=colors.HexColor(\'#183B50\'),spaceAfter=15))\n    st.add(ParagraphStyle(name=\'SubGS\',parent=st[\'Heading2\'],textColor=colors.HexColor(\'#183B50\'),spaceBefore=14,spaceAfter=7))\n    doc=SimpleDocTemplate(str(p),pagesize=A4,leftMargin=48,rightMargin=48,topMargin=47,bottomMargin=47)\n    h=lambda s:Paragraph(s,st[\'SubGS\']);t=lambda s:Paragraph(s,st[\'CorpoGS\'])\n    blocos=[Paragraph(\'Radar Cidadão — Fábrica de Corpus\',st[\'TitGS\']), t(\'<b>Relatório técnico — rascunho sujeito à revisão da equipe.</b> Não declarar este PDF final enquanto houver campos pendentes.\'),\n        h(\'1. Problema, público e ODS\'),t(\'Tema: prevenção de riscos associados a enchentes, chuvas intensas e deslizamentos. Persona: morador de área suscetível a desastres, que precisa de orientação pública em linguagem acessível. ODS relacionado: 11 (Cidades e Comunidades Sustentáveis).\'),\n        h(\'2. Documentos e ingestão\'),t(\'Fontes: documentos de órgãos públicos da área de saúde, cidades e proteção civil. <b>PENDENTE:</b> listar os arquivos reais utilizados, URLs, datas e número total de páginas. A ferramenta deve demonstrar PDF, TXT, DOCX, HTML e texto colado.\'),\n        h(\'3. Processamento\'),t(\'Esteira Python: extração de conteúdo com rastreabilidade; limpeza de ruído; normalização em UTF-8; anonimização anterior à geração; segmentação preservando frases e parágrafos. <b>PENDENTE:</b> anexar um exemplo comprovado de antes/depois da anonimização e medir o tamanho dos segmentos entre 256 e 512 tokens.\'),\n        h(\'4. Geração e estatísticas\')]\n    data=[[\'Corpus\',\'Válidos\',\'Mínimo\',\'Média aprox. tokens\',\'Duplicatas\']]\n    for k in (\'texto\',\'qa\',\'chat\'):\n        v=met[k];data.append([k,str(v[\'quantidade\']),str(v[\'minimo\']),str(v[\'media_tokens_aprox\']),str(v[\'duplicatas_removidas\'])])\n    tb=Table(data,colWidths=[90,65,65,130,85],repeatRows=1,hAlign=\'LEFT\');tb.setStyle(TableStyle([\n        (\'BACKGROUND\',(0,0),(-1,0),colors.HexColor(\'#183B50\')),(\'TEXTCOLOR\',(0,0),(-1,0),colors.white),(\'ROWBACKGROUNDS\',(0,1),(-1,-1),[colors.white,colors.HexColor(\'#F0F5F7\')]),(\'VALIGN\',(0,0),(-1,-1),\'MIDDLE\'),(\'TOPPADDING\',(0,0),(-1,-1),9),(\'BOTTOMPADDING\',(0,0),(-1,-1),9)]))\n    blocos += [tb,Spacer(1,12),t(f"Recusas estruturais: <b>{met[\'recusas_validas\']}</b> (mínimo: 10). As médias de tokens são <b>estimativas lexicais</b>, não medições com o tokenizador do modelo. Cinco amostras aleatórias de cada corpus foram salvas em <b>amostras_5_por_corpus.json</b>."),\n        h(\'5. Qualidade, riscos e LGPD\'),t(\'O projeto utiliza Qwen2.5-14B-Instruct para auxiliar a geração, com prompt solicitando apoio estrito no texto-fonte, clareza e respeito a quem executa a ação. Filtros automáticos removem entradas inválidas e duplicatas exatas. Os filtros NÃO comprovam fidelidade factual. <b>PENDENTE:</b> equipe revisar pelo menos 20% dos exemplos de IA em revisao_humana_20_porcento.csv e conferir as fontes; corrigir ou excluir exemplos inadequados antes da entrega.\'),\n        h(\'6. Limitações e dificuldades\'),t(\'Desafios observados: respostas repetidas, troca entre responsabilidade do morador e de órgãos públicos, respostas sobre saúde incompletas e divergência entre juiz automático e revisão humana. Exemplos de recusa foram acrescentados para pedidos fora de escopo. Nenhuma taxa de qualidade humana deve ser declarada antes da revisão.\'),\n        h(\'7. Reprodutibilidade e demonstração\'),t(\'Código e arquivos deverão ser disponibilizados em um único link público/compartilhado. <b>PENDENTE:</b> inserir nomes completos da equipe, confirmar execução de ponta a ponta e gravar vídeo de até 5 minutos mostrando upload, processamento, geração dos três JSONL, estatísticas e exemplos.\')]\n    doc.build(blocos)\n    return p\nif __name__==\'__main__\':\n    import sys\n    print(construir(sys.argv[1]))\n',encoding='utf-8')
import fechar_entrega as fechar
importlib.reload(fechar)
existentes=fechar.ler_jsonl(ANTERIOR/'corpus_qa.jsonl')
validos,dups,invalidos=fechar.filtrar(existentes,'qa')
print('QA existentes válidas:',len(validos),'| duplicatas:',dups,'| inválidas:',len(invalidos))
assert len(validos)>=90, 'Quantidade inesperada; não sobrescrever a entrega anterior.'
print('Chat existentes:',len(fechar.ler_jsonl(ANTERIOR/'corpus_chat.jsonl')))
print('Destino corrigido (ainda NÃO criado):',NOVO)

In [ ]:
# 2. Segmentação real e modelo Qwen (somente se for necessário gerar exemplos)
import os
os.chdir(PROJETO)
from fabrica_corpus.ingestao import ingerir_pasta
from fabrica_corpus.limpeza import limpar_corpus
from fabrica_corpus.anonimizacao import anonimizar_corpus
from fabrica_corpus.segmentacao import segmentar
from fabrica_corpus import geracao
geracao=importlib.reload(geracao)
assert getattr(geracao,'VERSAO_QA_CHAT',None)=='v8_3', ('É necessário carregar a geração v8.3 antes do complemento. Versão encontrada: '+str(getattr(geracao,'VERSAO_QA_CHAT',None)))
limpos,_=limpar_corpus(ingerir_pasta('dados/entrada'))
anon,_=anonimizar_corpus(limpos)
segs,_=segmentar(anon)
print('Segmentos do projeto:',len(segs))
assert len(segs)>=50
import torch
if len(validos)<100:
    assert torch.cuda.is_available(), 'Ative GPU no Colab (Ambiente de execução > Alterar tipo de ambiente) para rodar Qwen.'
    if 'gerar' not in globals() or not callable(gerar):
        gerar=geracao.carregar_qwen()
CACHE=BASE/'cache_geracao'
CACHE.mkdir(parents=True,exist_ok=True)
print('Versão QA:',geracao.VERSAO_QA_CHAT,'| Cache:',CACHE)

In [ ]:
# 3. Gerar somente novas QAs, filtrar de novo e exigir mínimo de 100
# Os segmentos já usados pelas 95 QAs são excluídos de preferência.
# Não há inserção artificial de exemplos nem desligamento do juiz.
import copy
qa_ok=list(validos)
segmentos_ja_usados={e.get('segmento_id') for e in qa_ok if e.get('segmento_id')}
candidatos=[s for s in segs if s.id not in segmentos_ja_usados]
print('Segmentos inéditos disponíveis:',len(candidatos))
assert candidatos or len(qa_ok)>=100, 'Sem segmentos inéditos; é preciso revisar a base.'
novos_brutos=[]
for rodada in range(1,5):
    if len(qa_ok)>=100: break
    print(f'\nRODADA {rodada}: {len(qa_ok)}/100 QAs válidas. Gerando candidatas com filtros v8.3...')
    # Seleção diversa com semente fixa, sem mexer nos documentos ou cache.
    pool=list(candidatos)
    random.Random(2026+rodada).shuffle(pool)
    pool=pool[:min(len(pool),90)]
    if not pool: break
    geradas,estat=geracao.gerar_qa(pool,gerar,total=16,por_segmento=2,pasta_cache=CACHE,filtrar_repeticoes=True)
    novos_brutos+=geradas
    normalizadas=fechar.normalizar_qa(geradas)
    combinados,dup,ruins=fechar.filtrar(qa_ok+normalizadas,'qa')
    qa_ok=combinados
    candidatos=[s for s in candidatos if s.id not in {e.get('segmento_id') for e in normalizadas}]
    print('Acumulado válido:',len(qa_ok),'| duplicatas removidas nesta rodada:',dup,'| inválidas:',len(ruins))
    print('Geração:',estat.get('exemplos',0),'aceitas pelo gerador; motivos de descarte internos:',estat.get('motivos_descarte',{}))
assert len(qa_ok)>=100, (f'Geradas apenas {len(qa_ok)} QAs válidas. NÃO exportei pacote incompleto. Reexecute esta célula com GPU e cache disponíveis ou revise as fontes.')
qa_ok=qa_ok[:100]
print('META ALCANÇADA:',len(qa_ok),'QAs únicas e estruturalmente válidas.')

In [ ]:
# 4. Exportar pacote CORRIGIDO sem apagar os originais nem fingir revisão humana
import statistics, hashlib, datetime, zipfile, csv
# Protege o trabalho humano prévio: não substitui pasta corrigida se a revisão já foi preenchida.
if NOVO.exists() and (NOVO/'revisao_humana_20_porcento.csv').exists():
    rev=NOVO/'revisao_humana_20_porcento.csv'
    with rev.open(encoding='utf-8-sig',newline='') as f:
        linhas=list(csv.DictReader(f,delimiter=';'))
    if any(r.get('revisor','').strip() or r.get('apoio_na_fonte_S_N','').strip() for r in linhas):
        raise RuntimeError('Há revisão humana preenchida na pasta corrigida. Não sobrescrever: faça backup/reconcilie antes.')
NOVO.mkdir(parents=True,exist_ok=True)
for nome in ('corpus_texto.jsonl','corpus_chat.jsonl'):
    shutil.copy2(ANTERIOR/nome,NOVO/nome)
fechar.salvar_jsonl(NOVO/'corpus_qa.jsonl',qa_ok)
metrics={}
for tipo in ('texto','qa','chat'):
    data=fechar.ler_jsonl(NOVO/f'corpus_{tipo}.jsonl')
    limpa,d,ruins=fechar.filtrar(data,tipo)
    assert len(data)==len(limpa) and len(data)>=fechar.MINIMOS[tipo], f'{tipo} não atingiu a meta'
    anterior_metricas=json.loads((ANTERIOR/'estatisticas.json').read_text(encoding='utf-8'))
    metrics[tipo]={'quantidade':len(data),'minimo':fechar.MINIMOS[tipo], 'quantidade_atingida':True,
      'media_tokens_aprox':round(statistics.mean(fechar.estimar_tokens(e) for e in data),1),
      'duplicatas_removidas':anterior_metricas.get(tipo,{}).get('duplicatas_removidas',0) if tipo!='qa' else anterior_metricas.get('qa',{}).get('duplicatas_removidas',0)+len(novos_brutos)-(len(qa_ok)-len(validos)),
      'invalidos_removidos':anterior_metricas.get(tipo,{}).get('invalidos_removidos',0)}
metrics['recusas_validas']=sum(e.get('tipo')=='recusa' for e in fechar.ler_jsonl(NOVO/'corpus_chat.jsonl'))
metrics['recusas_atingidas']=metrics['recusas_validas']>=10
assert metrics['recusas_atingidas']
metrics.update(revisao_humana_confirmada=False,fontes_documentais_conferidas=False,pronto_para_entrega=False,
 observacao='Estrutura validada; revisão humana >=20%, fidelidade às fontes, README e vídeo pendentes.')
(NOVO/'estatisticas.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8')
amostras={tipo:random.Random(2026).sample(fechar.ler_jsonl(NOVO/f'corpus_{tipo}.jsonl'),5) for tipo in ('texto','qa','chat')}
(NOVO/'amostras_5_por_corpus.json').write_text(json.dumps(amostras,ensure_ascii=False,indent=2),encoding='utf-8')
fechar.criar_revisao(NOVO)
# Leva as anotações anteriores para a conferência, mas não as marca automaticamente como revisadas.
if (ANTERIOR/'revisao_humana_20_porcento.csv').exists():
    shutil.copy2(ANTERIOR/'revisao_humana_20_porcento.csv',NOVO/'revisao_humana_ANTERIOR_PARA_CONSULTA.csv')
(NOVO/'AVISO_PENDENCIAS.txt').write_text('RASCUNHO: revisar >=20% dos exemplos com fontes originais, conferir formatos e páginas, fechar README e gravar vídeo <=5 minutos. Não declarar entrega aprovada automaticamente.\n',encoding='utf-8')
# Reaproveitar materiais de documentação existentes; sem marcar revisão aprovada.
for nome in ('README_ENTREGA.md','ROTEIRO_VIDEO_5_MIN.md'):
    arquivos=[Path('/content')/nome, PROJETO/nome, ANTERIOR/nome]
    origem=next((p for p in arquivos if p.exists()),None)
    if origem: shutil.copy2(origem,NOVO/nome)
subprocess.run([sys.executable,'-m','pip','-q','install','reportlab'],check=True)
from relatorio_pdf import construir
relatorio=construir(NOVO)
ret=fechar.auditar_pasta(NOVO)
print(json.dumps(ret,ensure_ascii=False,indent=2))
assert all(ret[t]['ok'] for t in ('texto','qa','chat')) and ret['recusas']>=10
ARQUIVO=BASE/'RADAR_CIDADAO_NLP_CORRIGIDO_RASCUNHO.zip'
with zipfile.ZipFile(ARQUIVO,'w',zipfile.ZIP_DEFLATED) as z:
    for arquivo in sorted(NOVO.rglob('*')):
        if arquivo.is_file(): z.write(arquivo,arcname=arquivo.relative_to(BASE))
print('\nPACOTE:',ARQUIVO)
print('REVISÃO HUMANA:',NOVO/'revisao_humana_20_porcento.csv')
print('RELATÓRIO:',relatorio)
print('ATENÇÃO: quantidades corrigidas, mas revisão humana e vídeo continuam pendentes.')